# MagicFinance — Module A: MagicReddit

**Goal**: Score Reddit stock recommendations using Qwen 9B.  
Posts from `r/ValueInvesting`, `r/SecurityAnalysis`, `r/stocks` are fetched live,  
scored on 5 quality dimensions, and stored in Qdrant for downstream modules.

**Pipeline**:
1. Fetch posts from Reddit API (live)
2. Filter posts with detected tickers
3. Score each post with Qwen 9B → structured 7-field signal
4. Store high-quality signals in Qdrant
5. Fire Slack alert for `confidence_level >= 0.75`
6. Visualise signal distribution

**Output**: Scored signals stored in Qdrant collection `magicfinance_reddit_signals`

## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s: %(message)s')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from datetime import datetime
from IPython.display import display

from magicfinance import config
from magicfinance.reddit_client import fetch_all_subreddits, filter_posts_with_tickers
from magicfinance.llm_client import score_reddit_post, check_ollama_health
from magicfinance import qdrant_client as qc
from magicfinance.slack_client import alert_high_confidence_signal

print(f'MagicFinance v{config.__version__ if hasattr(config, "__version__") else "1.0.0"}')
print(f'Qdrant: {config.QDRANT_HOST}:{config.QDRANT_PORT}')
print(f'LLM: {config.MODEL_9B} via {config.OLLAMA_BASE_URL}')

## 1. Health Checks

In [ ]:
# Check Ollama is running
ollama_ok = check_ollama_health()
print(f'Ollama reachable: {"✅" if ollama_ok else "❌ — start with: ollama serve"}')

# Check Qdrant (via Tailscale)
try:
    client = qc.get_client()
    qc.ensure_collections()
    print('Qdrant: ✅')
except Exception as e:
    print(f'Qdrant: ❌ — {e}')
    print('Make sure Tailscale is connected: tailscale up')

## 2. Fetch Reddit Posts

In [ ]:
# Fetch posts from all 3 subreddits
print(f'Fetching posts from: {config.SUBREDDITS}...')
all_posts = fetch_all_subreddits(
    subreddits=config.SUBREDDITS,
    limit=config.REDDIT_POST_LIMIT,
    min_upvotes=config.MIN_UPVOTES,
)

posts_with_tickers = filter_posts_with_tickers(all_posts)

print(f'Total posts fetched: {len(all_posts)}')
print(f'Posts with tickers: {len(posts_with_tickers)}')

# Show sample
df_posts = pd.DataFrame(posts_with_tickers)
display(df_posts[['subreddit','title','score','detected_tickers','word_count']].head(10))

## 3. Score Posts with Qwen 9B

In [ ]:
# Score each post — this takes ~30-60s per post depending on Qwen 9B speed
# Adjust SCORE_LIMIT to control how many posts to score in this session
SCORE_LIMIT = 20  # increase for a full run

scored_signals = []
errors = []

posts_to_score = posts_with_tickers[:SCORE_LIMIT]
print(f'Scoring {len(posts_to_score)} posts with {config.MODEL_9B}...')

for i, post in enumerate(posts_to_score):
    try:
        full_text = f"{post['title']}\n\n{post['selftext']}"
        score = score_reddit_post(full_text, post['detected_tickers'])
        
        # Build signal object
        primary_ticker = score.get('primary_ticker') or (post['detected_tickers'][0] if post['detected_tickers'] else None)
        if not primary_ticker:
            continue
            
        signal = {
            'ticker': primary_ticker.upper(),
            'thesis_score': score.get('thesis_score', 0),
            'risk_acknowledgment': score.get('risk_acknowledgment', 0),
            'data_quality': score.get('data_quality', 0),
            'specificity': score.get('specificity', 0),
            'original_thinking': score.get('original_thinking', 0),
            'confidence_level': score.get('confidence_level', 0),
            'is_investable': score.get('is_investable', False),
            'explanation': score.get('explanation', {}),
            'source_subreddit': post['subreddit'],
            'post_id': post['id'],
            'post_title': post['title'][:100],
            'post_score': post['score'],
            'signal_timestamp': datetime.utcnow().isoformat(),
        }
        scored_signals.append(signal)
        print(f'  [{i+1}/{len(posts_to_score)}] {primary_ticker}: confidence={signal["confidence_level"]:.2f} investable={signal["is_investable"]}')
        
    except Exception as e:
        errors.append({'post_id': post['id'], 'error': str(e)})
        print(f'  [{i+1}/{len(posts_to_score)}] ERROR: {e}')

print(f'\nScored: {len(scored_signals)} signals, {len(errors)} errors')

## 4. Store in Qdrant + Fire Slack Alerts

In [ ]:
alerts_fired = 0

for signal in scored_signals:
    # Store all signals in Qdrant (even low confidence — for research)
    qc.upsert_reddit_signal(signal)
    
    # Fire Slack alert only for high-confidence signals
    if signal['confidence_level'] >= config.CONFIDENCE_THRESHOLD:
        alert_high_confidence_signal(signal)
        alerts_fired += 1

print(f'Stored {len(scored_signals)} signals in Qdrant')
print(f'Slack alerts fired: {alerts_fired} (threshold: confidence >= {config.CONFIDENCE_THRESHOLD})')

## 5. Analysis: Signal Distribution

In [ ]:
# Load all signals from Qdrant (including previous sessions)
all_signals = qc.get_all_signals(limit=500)
df = pd.DataFrame(all_signals)

print(f'Total signals in Qdrant: {len(df)}')
print(f'Investable: {df["is_investable"].sum()} ({df["is_investable"].mean():.0%})')
print(f'High confidence (>={config.CONFIDENCE_THRESHOLD}): {(df["confidence_level"] >= config.CONFIDENCE_THRESHOLD).sum()}')
print()
display(df.groupby('source_subreddit')['confidence_level'].describe().round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('MagicReddit — Signal Quality Distribution', fontsize=14, fontweight='bold')

# Confidence distribution
axes[0].hist(df['confidence_level'], bins=20, color='steelblue', edgecolor='white')
axes[0].axvline(config.CONFIDENCE_THRESHOLD, color='red', linestyle='--', label=f'Threshold ({config.CONFIDENCE_THRESHOLD})')
axes[0].set_title('Confidence Level Distribution')
axes[0].set_xlabel('Confidence Level')
axes[0].legend()

# Scores by dimension
score_cols = ['thesis_score','risk_acknowledgment','data_quality','specificity','original_thinking']
score_means = df[score_cols].mean()
axes[1].barh(score_cols, score_means, color='teal')
axes[1].set_title('Average Scores by Dimension')
axes[1].set_xlim(0, 1)

# Investable vs not by subreddit
if 'source_subreddit' in df.columns:
    sub_investable = df.groupby('source_subreddit')['is_investable'].mean()
    axes[2].bar(sub_investable.index, sub_investable.values, color='darkorange')
    axes[2].set_title('Investable Rate by Subreddit')
    axes[2].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))

plt.tight_layout()
plt.savefig('../data/module_a_signals.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Top Investable Signals

In [ ]:
investable = df[df['is_investable'] == True].sort_values('confidence_level', ascending=False)

display_cols = ['ticker','confidence_level','thesis_score','risk_acknowledgment','data_quality','source_subreddit']
print(f'Top investable signals ({len(investable)} total):')
display(investable[display_cols].head(15).style.format({
    'confidence_level': '{:.0%}',
    'thesis_score': '{:.0%}',
    'risk_acknowledgment': '{:.0%}',
    'data_quality': '{:.0%}',
}).background_gradient(subset=['confidence_level'], cmap='Greens'))

print('\n✅ Module A complete. Run notebook_d.ipynb next.')